# Modal single-GPU memory-footprint experiment

This experiment measures peak H200 CUDA memory for the current uncheckpointed
Diffusion-ARC single-GPU path as a function of exact FLUX conditioning length
and physical FLUX microbatch size.

A successful run keeps NVARC and the VAE frozen, keeps complete FLUX parameters,
gradients, and AdamW moments in FP32, executes FLUX under BF16 autocast, and
proves that every measured optimizer step leaves FLUX weights unchanged by
forcing the effective learning rate to zero.

Before running:

- select one H200;
- attach the existing diffusion-arc-models Volume at
  /mnt/diffusion-arc-models with write access;
- attach a Modal Secret whose key is GITHUB_TOKEN;
- start from a fresh kernel and run top-to-bottom.

The notebook performs no model or dataset download. Expected OOM results are
recorded rather than treated as experimental failures when CUDA recovers to the
steady-state baseline.


In [1]:
!nvidia-smi


Tue Aug 25 05:42:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200                    On  |   00000000:A4:00.0 Off |                    0 |
| N/A   37C    P0             82W /  700W |       4MiB / 143771MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
%uv pip uninstall torchvision


Using Python 3.12.6 environment at: /usr/local
Uninstalled 1 package in 224ms
 - torchvision==0.23.0+cu129
Note: you may need to restart the kernel to use updated packages.


## Experiment plan

Hypothesis: after the fixed model, gradient, and AdamW footprint is resident,
peak memory grows mainly with the product of physical microbatch size and the
joint text-plus-900-image-token length.

Variables:

- exact text lengths: 128, 256, 512, 1,024, 2,048, 4,096, and 8,192;
- physical FLUX microbatch sizes: powers of two from 1 through 64;
- activation checkpointing: disabled for every case.

Measurements:

- actual single-sequence NVARC conditioning peak for every text length;
- transient FLUX microbatch-assembly peak;
- steady-state forward, backward, and AdamW-step peak with FP32 moments resident;
- allocated and reserved CUDA memory, diagnostic step time, PASS/OOM stage, and
  a conservative 90%-of-VRAM classification.

The sweep stops trying larger physical batches after the first OOM for one text
length. A recovered OOM is still only a provisional boundary; a later canonical
capacity decision should rerun the neighboring cases in fresh kernels.


## 1. Fix experiment identity and durable reporting


In [3]:
from __future__ import annotations

import base64
from datetime import datetime, timezone
import gc
import hashlib
import importlib
import importlib.metadata
import json
import math
import os
from pathlib import Path
import platform
import re
import subprocess
import sys
import time

SEED = 167
NOTEBOOK_PROCEDURE_VERSION = 1
SOURCE_COMMIT = '00c978ff4bddef8a7f1ca0bbc224c0d88cd972d1'
REPOSITORY_URL = 'https://github.com/joshuakauling/diffusion-arc.git'
SOURCE_DIR = Path('/root/diffusion-arc')

SEQUENCE_LENGTHS = (128, 256, 512, 1024, 2048, 4096, 8192)
FLUX_MICROBATCH_SIZES = (1, 2, 4, 8, 16, 32, 64)
ACTIVATION_CHECKPOINTING = False
SAFE_RESERVED_FRACTION = 0.90
OOM_RECOVERY_TOLERANCE_GIB = 0.25
MINIMUM_VRAM_GIB = 120.0

EXPECTED_FLUX_REPO_ID = 'black-forest-labs/FLUX.2-klein-base-4B'
EXPECTED_FLUX_REVISION = '8c44a2fbef88fae175da65df054db0f901aa9747'
EXPECTED_NVARC_HANDLE = (
    'sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1'
)
EXPECTED_PROJECT_SHA256 = {
    'flux_transformer': (
        '9c5fed22b76baea749d88fc2abe3ad53245e7b21a0d353a762665eea00043b92'
    ),
    'diffusers_vae': (
        'ca70d2202afe6415bdbcb8793ba8cd99fd159cfe6192381504d6c4d3036e0f04'
    ),
}
EXPECTED_NVARC_FILE_SIZES = {
    'model-00001-of-00002.safetensors': 4_996_836_472,
    'model-00002-of-00002.safetensors': 2_270_397_024,
}

VOLUME_ROOT = Path('/mnt/diffusion-arc-models')
FLUX_CHECKPOINT = VOLUME_ROOT / 'flux2-klein-base-4b'
NVARC_CHECKPOINT = VOLUME_ROOT / 'nvarc-qwen3-4b-grids15-sft139'
DOWNLOAD_MANIFEST = VOLUME_ROOT / 'download-manifest.json'
RESULTS_ROOT = VOLUME_ROOT / 'results' / 'single-gpu-memory-footprint'

if tuple(sorted(SEQUENCE_LENGTHS)) != SEQUENCE_LENGTHS:
    raise ValueError('SEQUENCE_LENGTHS must be strictly increasing')
if SEQUENCE_LENGTHS[-1] != 8192:
    raise ValueError('The sweep must include the exact 8,192-token maximum')
if any(length & (length - 1) for length in SEQUENCE_LENGTHS):
    raise ValueError('Every sequence length must be a power of two')
if tuple(sorted(FLUX_MICROBATCH_SIZES)) != FLUX_MICROBATCH_SIZES:
    raise ValueError('FLUX_MICROBATCH_SIZES must be strictly increasing')
if ACTIVATION_CHECKPOINTING:
    raise ValueError('This approved experiment requires checkpointing disabled')
if re.fullmatch(r'[0-9a-f]{40}', SOURCE_COMMIT) is None:
    raise ValueError('SOURCE_COMMIT must be a full 40-character Git SHA')
if not VOLUME_ROOT.is_dir():
    raise RuntimeError(f'Attach the model Volume at {VOLUME_ROOT} first')

write_probe = VOLUME_ROOT / '.memory-footprint-write-probe'
try:
    write_probe.write_text('ok\n', encoding='utf-8')
    write_probe.unlink()
except OSError as error:
    raise RuntimeError(f'Modal Volume is not writable: {VOLUME_ROOT}') from error

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
REPORT_DIR = RESULTS_ROOT / RUN_ID
REPORT_PATH = REPORT_DIR / 'report.json'
REPORT_DIR.mkdir(parents=True, exist_ok=False)

REPORT: dict[str, object] = {
    'schema_version': 1,
    'status': 'RUNNING',
    'run_id': RUN_ID,
    'notebook_procedure_version': NOTEBOOK_PROCEDURE_VERSION,
    'question': (
        'How do exact NVARC conditioning length and physical FLUX microbatch '
        'size affect steady-state H200 peak memory without activation '
        'checkpointing?'
    ),
    'configuration': {
        'seed': SEED,
        'sequence_lengths': list(SEQUENCE_LENGTHS),
        'flux_microbatch_sizes': list(FLUX_MICROBATCH_SIZES),
        'activation_checkpointing': ACTIVATION_CHECKPOINTING,
        'optimizer_learning_rate_override': 0.0,
        'safe_reserved_fraction': SAFE_RESERVED_FRACTION,
        'minimum_vram_gib': MINIMUM_VRAM_GIB,
    },
    'paths': {
        'volume_root': str(VOLUME_ROOT),
        'flux_checkpoint': str(FLUX_CHECKPOINT),
        'nvarc_checkpoint': str(NVARC_CHECKPOINT),
        'download_manifest': str(DOWNLOAD_MANIFEST),
        'report': str(REPORT_PATH),
    },
    'conditioning': [],
    'cases': [],
    'failures': [],
}


def save_report() -> None:
    temporary_path = REPORT_PATH.with_suffix('.json.tmp')
    temporary_path.write_text(
        json.dumps(REPORT, indent=2, sort_keys=True, allow_nan=False) + '\n',
        encoding='utf-8',
    )
    temporary_path.replace(REPORT_PATH)
    if hasattr(os, 'sync'):
        os.sync()


def record_failure(stage: str, error: BaseException) -> None:
    failure = {
        'stage': stage,
        'type': type(error).__name__,
        'message': str(error),
        'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    failures = REPORT.setdefault('failures', [])
    if isinstance(failures, list):
        failures.append(failure)
    REPORT['status'] = 'FAIL'
    REPORT['failed_stage'] = stage
    save_report()


required_paths = {
    'FLUX checkpoint': FLUX_CHECKPOINT,
    'NVARC checkpoint': NVARC_CHECKPOINT,
    'download manifest': DOWNLOAD_MANIFEST,
}
missing_paths = {
    name: str(path) for name, path in required_paths.items() if not path.exists()
}
if missing_paths:
    error = FileNotFoundError(f'Missing model artifacts: {missing_paths}')
    record_failure('volume-artifacts', error)
    raise error

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

save_report()
print(f'Run ID: {RUN_ID}')
print(f'Durable report: {REPORT_PATH}')
print(f'Sequence lengths: {SEQUENCE_LENGTHS}')
print(f'Physical microbatches: {FLUX_MICROBATCH_SIZES}')


Run ID: 20260825T054229658645Z
Durable report: /mnt/diffusion-arc-models/results/single-gpu-memory-footprint/20260825T054229658645Z/report.json
Sequence lengths: (128, 256, 512, 1024, 2048, 4096, 8192)
Physical microbatches: (1, 2, 4, 8, 16, 32, 64)


## 2. Fetch the exact source commit

The GitHub token is passed as a temporary Basic-auth header and is never placed
in a URL, file, notebook output, or report.


In [4]:
def run_command(
    label: str,
    command: list[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
) -> str:
    completed = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        tail = '\n'.join(completed.stdout.splitlines()[-40:])
        if tail:
            print(tail)
        raise RuntimeError(f'{label} failed with exit code {completed.returncode}')
    return completed.stdout.strip()


def current_source_head() -> str:
    return run_command(
        'resolve source HEAD',
        ['git', 'rev-parse', 'HEAD'],
        cwd=SOURCE_DIR,
    )


def authenticated_git_environment() -> dict[str, str]:
    github_token = os.environ.get('GITHUB_TOKEN', '').strip()
    if not github_token:
        raise RuntimeError(
            "Modal Secret key 'GITHUB_TOKEN' is missing or empty. "
            'Attach the Secret and restart the kernel.'
        )
    basic_credential = base64.b64encode(
        f'x-access-token:{github_token}'.encode('utf-8')
    ).decode('ascii')
    environment = os.environ.copy()
    environment.pop('GITHUB_TOKEN', None)
    environment['GIT_TERMINAL_PROMPT'] = '0'
    environment['GIT_CONFIG_COUNT'] = '1'
    environment['GIT_CONFIG_KEY_0'] = (
        'http.https://github.com/.extraheader'
    )
    environment['GIT_CONFIG_VALUE_0'] = (
        f'AUTHORIZATION: basic {basic_credential}'
    )
    return environment


try:
    git_environment = authenticated_git_environment()
    if SOURCE_DIR.exists() and not (SOURCE_DIR / '.git').is_dir():
        raise RuntimeError(f'{SOURCE_DIR} exists but is not the expected clone')
    if not SOURCE_DIR.exists():
        SOURCE_DIR.mkdir(parents=True)
        run_command('git init', ['git', 'init'], cwd=SOURCE_DIR)

    remotes = run_command(
        'list Git remotes',
        ['git', 'remote'],
        cwd=SOURCE_DIR,
    ).splitlines()
    if 'origin' not in remotes:
        run_command(
            'add Git origin',
            ['git', 'remote', 'add', 'origin', REPOSITORY_URL],
            cwd=SOURCE_DIR,
        )
    origin_url = run_command(
        'read Git origin',
        ['git', 'remote', 'get-url', 'origin'],
        cwd=SOURCE_DIR,
    )
    if origin_url != REPOSITORY_URL:
        raise RuntimeError(f'Unexpected Git origin: {origin_url}')

    tracked_changes = run_command(
        'inspect source worktree',
        ['git', 'status', '--porcelain', '--untracked-files=no'],
        cwd=SOURCE_DIR,
    )
    if tracked_changes:
        raise RuntimeError('Preserve tracked source edits before refreshing')

    run_command(
        'fetch pinned source',
        ['git', 'fetch', '--depth=1', 'origin', SOURCE_COMMIT],
        cwd=SOURCE_DIR,
        env=git_environment,
    )
    fetched_commit = run_command(
        'resolve fetched source',
        ['git', 'rev-parse', 'FETCH_HEAD'],
        cwd=SOURCE_DIR,
    )
    if fetched_commit != SOURCE_COMMIT:
        raise RuntimeError(f'Fetched source differs: {fetched_commit}')

    run_command(
        'checkout pinned source',
        ['git', 'checkout', '--detach', 'FETCH_HEAD'],
        cwd=SOURCE_DIR,
    )
    RESOLVED_SOURCE_COMMIT = current_source_head()
    if RESOLVED_SOURCE_COMMIT != SOURCE_COMMIT:
        raise RuntimeError('Checked-out source differs from SOURCE_COMMIT')
except Exception as error:
    record_failure('source-fetch', error)
    raise
finally:
    if 'git_environment' in globals():
        git_environment.clear()
        del git_environment

REPORT['source'] = {
    'repository': REPOSITORY_URL,
    'requested_commit': SOURCE_COMMIT,
    'resolved_commit': RESOLVED_SOURCE_COMMIT,
    'path': str(SOURCE_DIR),
    'authentication': 'github_pat_basic_header',
}
save_report()
print(f'Source ready at {RESOLVED_SOURCE_COMMIT[:12]} in {SOURCE_DIR}')


Source ready at 00c978ff4bdd in /root/diffusion-arc


## 3. Install and verify the pinned runtime


In [5]:
PINNED_PACKAGES = {
    'accelerate': '1.12.0',
    'diffusers': '0.39.0',
    'einops': '0.8.1',
    'Pillow': '11.3.0',
    'safetensors': '0.8.0',
    'torch': '2.13.0',
    'transformers': '4.56.1',
}


def installed_version(distribution: str) -> str | None:
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None


requirements = [
    f'{name}=={version}'
    for name, version in PINNED_PACKAGES.items()
    if installed_version(name) != version
]
if any(value.startswith('torch==') for value in requirements) and 'torch' in sys.modules:
    raise RuntimeError('Restart: a mismatched torch is already imported')

try:
    if requirements:
        print('Installing:', ' '.join(requirements))
        run_command(
            'install dependencies',
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '--quiet',
                '--disable-pip-version-check',
                '--no-input',
                '--no-cache-dir',
                *requirements,
            ],
        )
    run_command(
        'install Diffusion-ARC source',
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '--quiet',
            '--disable-pip-version-check',
            '--no-input',
            '--no-deps',
            '--no-build-isolation',
            '-e',
            str(SOURCE_DIR),
        ],
    )
except Exception as error:
    record_failure('dependency-install', error)
    raise

source_path = str((SOURCE_DIR / 'src').resolve())
if source_path not in sys.path:
    sys.path.insert(0, source_path)
importlib.invalidate_caches()
import diffusion_arc

package_path = Path(diffusion_arc.__file__).resolve()
if SOURCE_DIR.resolve() not in package_path.parents:
    error = RuntimeError(f'Unexpected diffusion_arc import: {package_path}')
    record_failure('source-import', error)
    raise error

INSTALLED_VERSIONS = {
    name: installed_version(name) for name in PINNED_PACKAGES
}
version_mismatches = {
    name: (expected, INSTALLED_VERSIONS[name])
    for name, expected in PINNED_PACKAGES.items()
    if INSTALLED_VERSIONS[name] != expected
}
if version_mismatches:
    error = RuntimeError(f'Package mismatch: {version_mismatches}')
    record_failure('dependency-contract', error)
    raise error

REPORT['packages'] = INSTALLED_VERSIONS
REPORT['source']['package_path'] = str(package_path)
save_report()
print(f'diffusion_arc: {package_path}')
INSTALLED_VERSIONS


Installing: accelerate==1.12.0 diffusers==0.39.0 einops==0.8.1 Pillow==11.3.0 safetensors==0.8.0 torch==2.13.0 transformers==4.56.1
diffusion_arc: /root/diffusion-arc/src/diffusion_arc/__init__.py


{'accelerate': '1.12.0',
 'diffusers': '0.39.0',
 'einops': '0.8.1',
 'Pillow': '11.3.0',
 'safetensors': '0.8.0',
 'torch': '2.13.0',
 'transformers': '4.56.1'}

## 4. Run repository unit tests


In [6]:
test_environment = os.environ.copy()
test_environment['PYTHONDONTWRITEBYTECODE'] = '1'
test_environment['PYTHONPATH'] = str(SOURCE_DIR / 'src')
test_started = time.perf_counter()
test_process = subprocess.run(
    [sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests'],
    cwd=SOURCE_DIR,
    env=test_environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)
test_seconds = time.perf_counter() - test_started
test_output_tail = '\n'.join(test_process.stdout.splitlines()[-30:])
print(test_output_tail)
REPORT['unit_tests'] = {
    'returncode': test_process.returncode,
    'seconds': test_seconds,
    'output_tail': test_output_tail,
}
if test_process.returncode != 0:
    error = RuntimeError(f'Unit tests returned {test_process.returncode}')
    record_failure('unit-tests', error)
    raise error
save_report()
print(f'Unit tests: PASS in {test_seconds:.2f}s')


Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
..................................................................................................................../usr/local/lib/python3.12/site-packages/torch/nn/functional.py:3006: UserWarning: Mismatch dtype between input and weight: input dtype = c10::BFloat16, weight dtype = float, Cannot dispatch to fused implementation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/layer_norm.cpp:346.)
  return torch.rms_norm(input, normalized_shape, weight, eps)
...........................
----------------------------------------------------------------------
Ran 143 tests in 7.054s

OK
Unit tests: PASS in 35.11s


## 5. Validate artifact provenance and the H200 CUDA contract

The benchmark trusts the already downloaded Volume but revalidates its manifest,
checkpoint locations, NVARC shard sizes, and the exact source/runtime identity.
No remote model metadata is requested.


In [7]:
try:
    manifest_bytes = DOWNLOAD_MANIFEST.read_bytes()
    manifest_payload = json.loads(manifest_bytes)
    manifest_paths = {
        'flux': Path(manifest_payload['flux']['path']).resolve(),
        'nvarc': Path(manifest_payload['nvarc']['path']).resolve(),
    }
    expected_paths = {
        'flux': FLUX_CHECKPOINT.resolve(),
        'nvarc': NVARC_CHECKPOINT.resolve(),
    }
    manifest_checks = {
        'paths': manifest_paths == expected_paths,
        'flux_repo_id': (
            manifest_payload['flux']['repo_id'] == EXPECTED_FLUX_REPO_ID
        ),
        'flux_requested_revision': (
            manifest_payload['flux']['requested_revision']
            == EXPECTED_FLUX_REVISION
        ),
        'flux_resolved_revision': (
            manifest_payload['flux']['resolved_revision']
            == EXPECTED_FLUX_REVISION
        ),
        'project_file_sha256': (
            manifest_payload['flux']['project_file_sha256']
            == EXPECTED_PROJECT_SHA256
        ),
        'nvarc_handle': (
            manifest_payload['nvarc']['kaggle_handle']
            == EXPECTED_NVARC_HANDLE
        ),
        'nvarc_version': manifest_payload['nvarc']['version'] == 1,
        'nvarc_weight_sizes': (
            manifest_payload['nvarc']['weight_file_bytes']
            == EXPECTED_NVARC_FILE_SIZES
        ),
    }
    nvarc_actual_sizes = {
        filename: (NVARC_CHECKPOINT / filename).stat().st_size
        for filename in EXPECTED_NVARC_FILE_SIZES
    }
    manifest_checks['nvarc_actual_weight_sizes'] = (
        nvarc_actual_sizes == EXPECTED_NVARC_FILE_SIZES
    )
    failed_manifest_checks = [
        name for name, passed in manifest_checks.items() if not passed
    ]
    if failed_manifest_checks:
        raise RuntimeError(
            f'Artifact manifest checks failed: {failed_manifest_checks}'
        )
except Exception as error:
    record_failure('artifact-provenance', error)
    raise

REPORT['artifacts'] = {
    'manifest_sha256': hashlib.sha256(manifest_bytes).hexdigest(),
    'checks': manifest_checks,
    'nvarc_actual_weight_sizes': nvarc_actual_sizes,
}
save_report()
print('Artifact manifest and checkpoint locations: PASS')


Artifact manifest and checkpoint locations: PASS


In [8]:
import torch

try:
    if importlib.metadata.version('torch') != PINNED_PACKAGES['torch']:
        raise RuntimeError('Imported torch differs from the project pin')
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable; select an H200 kernel')
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError('Selected CUDA device does not support BF16')

    DEVICE = torch.device('cuda:0')
    torch.cuda.set_device(DEVICE)
    GPU_PROPERTIES = torch.cuda.get_device_properties(DEVICE)
    if 'H200' not in GPU_PROPERTIES.name.upper():
        raise RuntimeError(f'Select an H200, received {GPU_PROPERTIES.name}')
    GPU_VRAM_GIB = GPU_PROPERTIES.total_memory / 2**30
    if GPU_VRAM_GIB < MINIMUM_VRAM_GIB:
        raise RuntimeError(
            f'Requires {MINIMUM_VRAM_GIB:.0f} GiB, received {GPU_VRAM_GIB:.1f}'
        )
except Exception as error:
    record_failure('cuda-contract', error)
    raise


def cuda_memory_snapshot(label: str) -> dict[str, object]:
    torch.cuda.synchronize(DEVICE)
    free_bytes, total_bytes = torch.cuda.mem_get_info(DEVICE)
    return {
        'label': label,
        'allocated_gib': torch.cuda.memory_allocated(DEVICE) / 2**30,
        'reserved_gib': torch.cuda.memory_reserved(DEVICE) / 2**30,
        'peak_allocated_gib': torch.cuda.max_memory_allocated(DEVICE) / 2**30,
        'peak_reserved_gib': torch.cuda.max_memory_reserved(DEVICE) / 2**30,
        'device_free_gib': free_bytes / 2**30,
        'device_total_gib': total_bytes / 2**30,
    }


def safe_cuda_memory_snapshot(label: str) -> dict[str, object]:
    try:
        return cuda_memory_snapshot(label)
    except Exception as error:
        return {
            'label': label,
            'unavailable': f'{type(error).__name__}: {error}',
        }


def cuda_cleanup() -> None:
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize(DEVICE)


torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
cuda_cleanup()
torch.cuda.reset_peak_memory_stats(DEVICE)

nvidia_smi = run_command(
    'query nvidia-smi',
    [
        'nvidia-smi',
        '--query-gpu=driver_version,name,memory.total',
        '--format=csv,noheader',
    ],
)
CUDA_ENVIRONMENT = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_runtime': torch.version.cuda,
    'nvidia_smi': nvidia_smi,
    'gpu_name': GPU_PROPERTIES.name,
    'gpu_vram_gib': GPU_VRAM_GIB,
    'compute_capability': list(torch.cuda.get_device_capability(DEVICE)),
    'bf16_supported': torch.cuda.is_bf16_supported(),
    'sdpa_policy': {
        'flash_enabled': torch.backends.cuda.flash_sdp_enabled(),
        'memory_efficient_enabled': torch.backends.cuda.mem_efficient_sdp_enabled(),
        'math_enabled': torch.backends.cuda.math_sdp_enabled(),
    },
}
REPORT['environment'] = CUDA_ENVIRONMENT
REPORT['memory'] = {
    'runtime_baseline': cuda_memory_snapshot('runtime-baseline')
}
save_report()
print(f'GPU: {GPU_PROPERTIES.name} | VRAM: {GPU_VRAM_GIB:.1f} GiB')
CUDA_ENVIRONMENT


GPU: NVIDIA H200 | VRAM: 139.8 GiB


{'python': '3.12.6',
 'torch': '2.13.0+cu130',
 'cuda_runtime': '13.0',
 'nvidia_smi': '580.95.05, NVIDIA H200, 143771 MiB',
 'gpu_name': 'NVIDIA H200',
 'gpu_vram_gib': 139.80084228515625,
 'compute_capability': [9, 0],
 'bf16_supported': True,
 'sdpa_policy': {'flash_enabled': True,
  'memory_efficient_enabled': True,
  'math_enabled': True}}

## 6. Load the production models with checkpointing disabled


In [9]:
from diffusion_arc.training.single_gpu_config import load_single_gpu_training_config
from diffusion_arc.training.single_gpu_setup import (
    build_flux_optimizer,
    load_single_gpu_models,
    prepare_single_gpu_device,
    seed_single_gpu_process,
)

TEMP_ROOT = Path('/tmp') / f'diffusion-arc-memory-{RUN_ID}'
TRAINING_CONFIG_PATH = TEMP_ROOT / 'single_gpu.json'
if TEMP_ROOT.exists():
    raise RuntimeError(f'Refusing to reuse temporary directory: {TEMP_ROOT}')
TEMP_ROOT.mkdir(parents=True)

run_config = {
    'schema_version': 1,
    'seed': SEED,
    'device': 'cuda:0',
    'models': {
        'nvarc_checkpoint': str(NVARC_CHECKPOINT),
        'flux_checkpoint': str(FLUX_CHECKPOINT),
        'nvarc_dtype': 'bfloat16',
        'vae_dtype': 'float32',
        'flux_parameter_dtype': 'float32',
        'flux_compute_dtype': 'bfloat16',
    },
    'data': {
        'split_directory': str(TEMP_ROOT / 'unused-split'),
        'view_name': 'unused',
        'shuffle': False,
        'num_workers': 0,
    },
    'loop': {
        'max_optimizer_steps': 1,
        'optimizer_batch_size': 1,
        'flux_microbatch_size': 1,
        'min_context_pairs_before_target': 1,
    },
    'optimizer': {
        'learning_rate': 1e-5,
        'beta1': 0.9,
        'beta2': 0.999,
        'epsilon': 1e-8,
        'weight_decay': 0.01,
    },
}
TRAINING_CONFIG_PATH.write_text(
    json.dumps(run_config, indent=2) + '\n',
    encoding='utf-8',
)
training_config = load_single_gpu_training_config(TRAINING_CONFIG_PATH)
device = prepare_single_gpu_device(training_config)
seed_single_gpu_process(training_config.seed)

cuda_cleanup()
torch.cuda.reset_peak_memory_stats(device)
try:
    model_load_started = time.perf_counter()
    models = load_single_gpu_models(training_config, device=device)
    torch.cuda.synchronize(device)
    model_load_seconds = time.perf_counter() - model_load_started
except Exception as error:
    record_failure('model-load', error)
    raise

if getattr(models.transformer, 'gradient_checkpointing', None) is not False:
    error = RuntimeError(
        'The approved benchmark requires gradient_checkpointing == False'
    )
    record_failure('checkpointing-contract', error)
    raise error


def module_parameter_summary(module: torch.nn.Module) -> dict[str, object]:
    parameters = tuple(module.parameters())
    return {
        'parameter_tensors': len(parameters),
        'parameters': sum(parameter.numel() for parameter in parameters),
        'trainable_parameters': sum(
            parameter.numel()
            for parameter in parameters
            if parameter.requires_grad
        ),
        'devices': sorted({str(parameter.device) for parameter in parameters}),
        'dtypes': sorted({str(parameter.dtype) for parameter in parameters}),
    }


MODEL_SUMMARIES = {
    'nvarc': module_parameter_summary(models.nvarc_runner),
    'vae': module_parameter_summary(models.vae_encoder),
    'flux': module_parameter_summary(models.transformer),
}
MODEL_CHECKS = {
    'nvarc_frozen': MODEL_SUMMARIES['nvarc']['trainable_parameters'] == 0,
    'vae_frozen': MODEL_SUMMARIES['vae']['trainable_parameters'] == 0,
    'flux_fully_trainable': (
        MODEL_SUMMARIES['flux']['trainable_parameters']
        == MODEL_SUMMARIES['flux']['parameters']
    ),
    'nvarc_bfloat16': MODEL_SUMMARIES['nvarc']['dtypes'] == ['torch.bfloat16'],
    'vae_float32': MODEL_SUMMARIES['vae']['dtypes'] == ['torch.float32'],
    'flux_float32': MODEL_SUMMARIES['flux']['dtypes'] == ['torch.float32'],
    'flux_compute_bfloat16': models.flux_compute_dtype == torch.bfloat16,
    'activation_checkpointing_disabled': (
        models.transformer.gradient_checkpointing is False
    ),
}
failed_model_checks = [
    name for name, passed in MODEL_CHECKS.items() if not passed
]
if failed_model_checks:
    error = RuntimeError(f'Model checks failed: {failed_model_checks}')
    record_failure('model-contract', error)
    raise error

model_load_memory = cuda_memory_snapshot('after-model-load')
REPORT['models'] = {
    'load_seconds': model_load_seconds,
    'summaries': MODEL_SUMMARIES,
    'checks': MODEL_CHECKS,
}
REPORT['memory']['after_model_load'] = model_load_memory
save_report()
print(f'Model load: {model_load_seconds:.2f}s')
print(MODEL_CHECKS)
model_load_memory


Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
`torch_dtype` is deprecated! Use `dtype` instead!


Model load: 27.06s
{'nvarc_frozen': True, 'vae_frozen': True, 'flux_fully_trainable': True, 'nvarc_bfloat16': True, 'vae_float32': True, 'flux_float32': True, 'flux_compute_bfloat16': True, 'activation_checkpointing_disabled': True}


{'label': 'after-model-load',
 'allocated_gib': 21.522281169891357,
 'reserved_gib': 21.564453125,
 'peak_allocated_gib': 21.522281169891357,
 'peak_reserved_gib': 21.564453125,
 'device_free_gib': 117.721923828125,
 'device_total_gib': 139.80084228515625}

## 7. Prepare one real target latent and initialize AdamW state without updates

The warmup executes one real forward, backward, and AdamW step at effective
learning rate zero. This creates the exact FP32 Adam moments needed for
steady-state memory measurements while a tracked FLUX parameter must remain
bitwise unchanged.


In [10]:
from diffusion_arc.arc_data.canvas import _render_valid_grid_canvas
from diffusion_arc.model import NVARC_CONTEXT_WIDTH, NVARC_MAX_TOKENS
from diffusion_arc.training.flux_loss import compute_flux_microbatch_loss
from diffusion_arc.training.flux_input import FluxInput, NvarcSequenceState
from diffusion_arc.training.flux_microbatch import build_flux_microbatch

if tuple(SEQUENCE_LENGTHS) != (128, 256, 512, 1024, 2048, 4096, 8192):
    raise RuntimeError('Sequence-length matrix changed unexpectedly')
if NVARC_MAX_TOKENS != 8192:
    raise RuntimeError(f'Unexpected NVARC maximum: {NVARC_MAX_TOKENS}')
if NVARC_CONTEXT_WIDTH != 7680:
    raise RuntimeError(f'Unexpected conditioning width: {NVARC_CONTEXT_WIDTH}')

target_grid = ((2, 2), (0, 2))
target_image = _render_valid_grid_canvas(target_grid).unsqueeze(0)
torch.cuda.reset_peak_memory_stats(device)
with torch.no_grad():
    clean_latent_gpu = models.vae_encoder(target_image)
torch.cuda.synchronize(device)
latent_preparation_memory = cuda_memory_snapshot('target-latent-preparation')
if tuple(clean_latent_gpu.shape) != (1, 900, 128):
    raise RuntimeError(
        f'Unexpected clean latent shape: {tuple(clean_latent_gpu.shape)}'
    )
if clean_latent_gpu.dtype != torch.float32:
    raise RuntimeError(f'Unexpected clean latent dtype: {clean_latent_gpu.dtype}')
CLEAN_LATENT_CPU = clean_latent_gpu[0].detach().cpu()
del clean_latent_gpu, target_image
cuda_cleanup()

REPORT['target_latent'] = {
    'shape': list(CLEAN_LATENT_CPU.shape),
    'dtype': str(CLEAN_LATENT_CPU.dtype),
    'cuda_memory': latent_preparation_memory,
}
save_report()


def build_contract_microbatch(
    hidden_states_cpu: torch.Tensor,
    batch_size: int,
):
    if hidden_states_cpu.device.type != 'cpu':
        raise ValueError('hidden_states_cpu must reside on CPU')
    if hidden_states_cpu.ndim != 2:
        raise ValueError('hidden_states_cpu must have shape [L, D]')
    if hidden_states_cpu.shape[1] != NVARC_CONTEXT_WIDTH:
        raise ValueError('hidden_states_cpu has an unexpected width')
    if hidden_states_cpu.dtype != torch.bfloat16:
        raise TypeError('hidden_states_cpu must use bfloat16')
    if isinstance(batch_size, bool) or not isinstance(batch_size, int):
        raise TypeError('batch_size must be an integer')
    if batch_size <= 0:
        raise ValueError('batch_size must be positive')

    sequence_length = hidden_states_cpu.shape[0]
    flux_inputs = []
    for _ in range(batch_size):
        sequence_state = NvarcSequenceState(
            hidden_states=hidden_states_cpu.to(device=device),
            pending_prefix_ends=(sequence_length,),
        )
        flux_inputs.append(
            FluxInput(
                nvarc_sequence_state=sequence_state,
                prefix_end=sequence_length,
                clean_latent=CLEAN_LATENT_CPU.to(device=device),
            )
        )

    microbatch = build_flux_microbatch(tuple(flux_inputs))
    expected_text_shape = (
        batch_size,
        sequence_length,
        NVARC_CONTEXT_WIDTH,
    )
    if tuple(microbatch.text_inputs.encoder_hidden_states.shape) != expected_text_shape:
        raise RuntimeError(
            'Constructed text shape differs: '
            f'{tuple(microbatch.text_inputs.encoder_hidden_states.shape)}'
        )
    if tuple(microbatch.clean_latents.shape) != (batch_size, 900, 128):
        raise RuntimeError(
            f'Constructed latent shape differs: {tuple(microbatch.clean_latents.shape)}'
        )
    return microbatch


def run_nvarc_template(sequence_length: int) -> tuple[dict[str, object], torch.Tensor]:
    if sequence_length not in SEQUENCE_LENGTHS:
        raise ValueError(f'Unapproved sequence length: {sequence_length}')

    cuda_cleanup()
    torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter()
    input_ids = torch.zeros(sequence_length, dtype=torch.int64)
    sequence_offsets = torch.tensor((0, sequence_length), dtype=torch.int64)
    output = models.nvarc_runner(input_ids, sequence_offsets)
    torch.cuda.synchronize(device)
    seconds = time.perf_counter() - started
    memory = cuda_memory_snapshot(f'nvarc-conditioning-l{sequence_length}')
    hidden_states = output.sequence_hidden_states[0]
    if tuple(hidden_states.shape) != (
        sequence_length,
        NVARC_CONTEXT_WIDTH,
    ):
        raise RuntimeError(
            f'Unexpected NVARC output shape: {tuple(hidden_states.shape)}'
        )
    if hidden_states.dtype != torch.bfloat16:
        raise RuntimeError(
            f'Unexpected NVARC output dtype: {hidden_states.dtype}'
        )
    hidden_states_cpu = hidden_states.detach().cpu()
    record = {
        'status': 'PASS',
        'sequence_length': sequence_length,
        'token_pattern': 'repeated_valid_token_id_0',
        'output_shape': list(hidden_states_cpu.shape),
        'output_dtype': str(hidden_states_cpu.dtype),
        'output_gib': (
            hidden_states_cpu.numel() * hidden_states_cpu.element_size() / 2**30
        ),
        'seconds': seconds,
        'cuda_memory': memory,
    }
    del output, hidden_states, input_ids, sequence_offsets
    cuda_cleanup()
    return record, hidden_states_cpu


warmup_conditioning, warmup_hidden_cpu = run_nvarc_template(128)
benchmark_optimizer = build_flux_optimizer(
    training_config,
    transformer=models.transformer,
)
for parameter_group in benchmark_optimizer.param_groups:
    parameter_group['lr'] = 0.0
if benchmark_optimizer.state:
    raise RuntimeError('AdamW state must be lazy before warmup')

tracked_parameter_name, tracked_parameter = next(
    iter(models.transformer.named_parameters())
)
tracked_parameter_index = 0
tracked_before = float(
    tracked_parameter.detach().reshape(-1)[tracked_parameter_index].item()
)

warmup_microbatch = build_contract_microbatch(warmup_hidden_cpu, 1)
benchmark_optimizer.zero_grad(set_to_none=True)
warmup_generator = torch.Generator(device=device).manual_seed(SEED)
torch.cuda.reset_peak_memory_stats(device)
warmup_started = time.perf_counter()
warmup_loss = compute_flux_microbatch_loss(
    models.transformer,
    warmup_microbatch,
    compute_dtype=models.flux_compute_dtype,
    generator=warmup_generator,
)
if not torch.isfinite(warmup_loss).item():
    raise RuntimeError('Warmup loss is non-finite')
warmup_loss.backward()
benchmark_optimizer.step()
torch.cuda.synchronize(device)
warmup_seconds = time.perf_counter() - warmup_started
warmup_memory = cuda_memory_snapshot('adam-state-initialization')
warmup_loss_value = float(warmup_loss.detach().item())

tracked_after = float(
    tracked_parameter.detach().reshape(-1)[tracked_parameter_index].item()
)
if tracked_after != tracked_before:
    raise RuntimeError(
        'LR=0 warmup changed a FLUX parameter: '
        f'{tracked_before} -> {tracked_after}'
    )

moment_tensors = [
    state[name]
    for state in benchmark_optimizer.state.values()
    for name in ('exp_avg', 'exp_avg_sq')
    if isinstance(state.get(name), torch.Tensor)
]
if not benchmark_optimizer.state or not moment_tensors:
    raise RuntimeError('Warmup did not create AdamW state')
if any(moment.dtype != torch.float32 for moment in moment_tensors):
    raise RuntimeError('AdamW moments must use FP32 storage')

REPORT['warmup'] = {
    'conditioning': warmup_conditioning,
    'seconds': warmup_seconds,
    'loss': warmup_loss_value,
    'tracked_parameter': tracked_parameter_name,
    'tracked_parameter_index': tracked_parameter_index,
    'parameter_before': tracked_before,
    'parameter_after': tracked_after,
    'parameter_unchanged': tracked_after == tracked_before,
    'optimizer_parameter_states': len(benchmark_optimizer.state),
    'adam_moment_tensors': len(moment_tensors),
    'adam_moment_dtype': 'torch.float32',
    'cuda_memory': warmup_memory,
}

benchmark_optimizer.zero_grad(set_to_none=True)
del warmup_loss, warmup_microbatch, warmup_hidden_cpu
cuda_cleanup()
torch.cuda.reset_peak_memory_stats(device)
STEADY_STATE_BASELINE = cuda_memory_snapshot('steady-state-adam-baseline')

REPORT['memory']['steady_state_adam_baseline'] = STEADY_STATE_BASELINE
save_report()
print('Warmup PASS: FP32 AdamW state resident, FLUX weights unchanged')
STEADY_STATE_BASELINE


Warmup PASS: FP32 AdamW state resident, FLUX weights unchanged


{'label': 'steady-state-adam-baseline',
 'allocated_gib': 50.46032667160034,
 'reserved_gib': 50.78125,
 'peak_allocated_gib': 50.46032667160034,
 'peak_reserved_gib': 50.78125,
 'device_free_gib': 88.268798828125,
 'device_total_gib': 139.80084228515625}

## 8. Run the memory matrix

Each case first constructs the exact physical FLUX input through the repository's
NvarcSequenceState and build_flux_microbatch path. Peak counters are then reset
with that microbatch resident before the measured forward, backward, and LR-0
AdamW step.


In [11]:
def run_memory_case(
    *,
    sequence_length: int,
    batch_size: int,
    hidden_states_cpu: torch.Tensor,
) -> dict[str, object]:
    benchmark_optimizer.zero_grad(set_to_none=True)
    cuda_cleanup()
    microbatch = None
    loss = None
    stage = 'microbatch-assembly'
    tracked_before_case = float(
        tracked_parameter.detach().reshape(-1)[tracked_parameter_index].item()
    )

    try:
        torch.cuda.reset_peak_memory_stats(device)
        assembly_started = time.perf_counter()
        microbatch = build_contract_microbatch(hidden_states_cpu, batch_size)
        torch.cuda.synchronize(device)
        assembly_seconds = time.perf_counter() - assembly_started
        assembly_memory = cuda_memory_snapshot(
            f'assembly-l{sequence_length}-b{batch_size}'
        )

        stage = 'training-step'
        torch.cuda.reset_peak_memory_stats(device)
        case_generator = torch.Generator(device=device).manual_seed(
            SEED + sequence_length + batch_size
        )
        step_started = time.perf_counter()
        loss = compute_flux_microbatch_loss(
            models.transformer,
            microbatch,
            compute_dtype=models.flux_compute_dtype,
            generator=case_generator,
        )
        if not torch.isfinite(loss).item():
            raise RuntimeError('Measured loss is non-finite')
        loss.backward()
        benchmark_optimizer.step()
        torch.cuda.synchronize(device)
        step_seconds = time.perf_counter() - step_started
        step_memory = cuda_memory_snapshot(
            f'training-step-l{sequence_length}-b{batch_size}'
        )

        tracked_after_case = float(
            tracked_parameter.detach().reshape(-1)[tracked_parameter_index].item()
        )
        if tracked_after_case != tracked_before_case:
            raise RuntimeError(
                'LR=0 benchmark changed a FLUX parameter: '
                f'{tracked_before_case} -> {tracked_after_case}'
            )

        peak_allocated_gib = max(
            assembly_memory['peak_allocated_gib'],
            step_memory['peak_allocated_gib'],
        )
        peak_reserved_gib = max(
            assembly_memory['peak_reserved_gib'],
            step_memory['peak_reserved_gib'],
        )
        return {
            'status': 'PASS',
            'sequence_length': sequence_length,
            'physical_microbatch_size': batch_size,
            'joint_tokens_per_microbatch': batch_size * (sequence_length + 900),
            'conditioning_tensor_gib': (
                batch_size
                * sequence_length
                * NVARC_CONTEXT_WIDTH
                * torch.tensor([], dtype=torch.bfloat16).element_size()
                / 2**30
            ),
            'activation_checkpointing': False,
            'optimizer_learning_rate': 0.0,
            'loss': float(loss.detach().item()),
            'parameter_unchanged': tracked_after_case == tracked_before_case,
            'assembly_seconds': assembly_seconds,
            'step_seconds': step_seconds,
            'assembly_cuda_memory': assembly_memory,
            'step_cuda_memory': step_memory,
            'case_peak_allocated_gib': peak_allocated_gib,
            'case_peak_reserved_gib': peak_reserved_gib,
            'within_safe_reserved_fraction': (
                peak_reserved_gib
                <= GPU_VRAM_GIB * SAFE_RESERVED_FRACTION
            ),
        }
    except torch.OutOfMemoryError as error:
        return {
            'status': 'OOM',
            'stage': stage,
            'sequence_length': sequence_length,
            'physical_microbatch_size': batch_size,
            'joint_tokens_per_microbatch': batch_size * (sequence_length + 900),
            'activation_checkpointing': False,
            'optimizer_learning_rate': 0.0,
            'error': str(error),
            'cuda_memory_at_oom': safe_cuda_memory_snapshot(
                f'oom-{stage}-l{sequence_length}-b{batch_size}'
            ),
        }
    finally:
        benchmark_optimizer.zero_grad(set_to_none=True)
        del microbatch, loss
        cuda_cleanup()


CASE_RESULTS: list[dict[str, object]] = []
CONDITIONING_RESULTS: list[dict[str, object]] = []
SWEEP_ABORTED = False

for sequence_length in SEQUENCE_LENGTHS:
    print(f'\nConditioning length {sequence_length:,}')
    try:
        conditioning_record, hidden_states_cpu = run_nvarc_template(
            sequence_length
        )
    except torch.OutOfMemoryError as error:
        conditioning_record = {
            'status': 'OOM',
            'sequence_length': sequence_length,
            'stage': 'nvarc-conditioning',
            'error': str(error),
            'cuda_memory_at_oom': safe_cuda_memory_snapshot(
                f'oom-nvarc-conditioning-l{sequence_length}'
            ),
        }
        hidden_states_cpu = None
        cuda_cleanup()

    CONDITIONING_RESULTS.append(conditioning_record)
    REPORT['conditioning'] = CONDITIONING_RESULTS
    save_report()

    if hidden_states_cpu is None:
        print('  NVARC conditioning: OOM; skipping FLUX cases')
        continue

    for batch_size in FLUX_MICROBATCH_SIZES:
        print(f'  physical microbatch {batch_size}: ', end='')
        case_result = run_memory_case(
            sequence_length=sequence_length,
            batch_size=batch_size,
            hidden_states_cpu=hidden_states_cpu,
        )
        CASE_RESULTS.append(case_result)
        REPORT['cases'] = CASE_RESULTS
        save_report()

        if case_result['status'] == 'PASS':
            print(
                f"PASS | allocated {case_result['case_peak_allocated_gib']:.2f} "
                f"GiB | reserved {case_result['case_peak_reserved_gib']:.2f} GiB"
            )
            continue

        recovery_memory = safe_cuda_memory_snapshot(
            f'post-oom-recovery-l{sequence_length}-b{batch_size}'
        )
        case_result['recovery_cuda_memory'] = recovery_memory
        recovered_allocated = recovery_memory.get('allocated_gib')
        baseline_allocated = STEADY_STATE_BASELINE['allocated_gib']
        recovered = (
            isinstance(recovered_allocated, (int, float))
            and abs(recovered_allocated - baseline_allocated)
            <= OOM_RECOVERY_TOLERANCE_GIB
        )
        case_result['cuda_recovered_to_baseline'] = recovered
        REPORT['cases'] = CASE_RESULTS
        save_report()
        print(f'OOM during {case_result["stage"]} | recovered={recovered}')

        if not recovered:
            SWEEP_ABORTED = True
        break

    del hidden_states_cpu
    cuda_cleanup()
    if SWEEP_ABORTED:
        print('Sweep aborted because CUDA did not recover to the baseline')
        break

print(f'\nAttempted {len(CASE_RESULTS)} FLUX cases')



Conditioning length 128
  physical microbatch 1: PASS | allocated 65.67 GiB | reserved 73.27 GiB
  physical microbatch 2: PASS | allocated 66.88 GiB | reserved 77.99 GiB
  physical microbatch 4: PASS | allocated 75.95 GiB | reserved 84.36 GiB
  physical microbatch 8: PASS | allocated 94.13 GiB | reserved 96.98 GiB
  physical microbatch 16: PASS | allocated 130.67 GiB | reserved 133.91 GiB
  physical microbatch 32: 

[W825 05:46:16.250619075 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 404750336 bytes (free: 7602176, total: 150110011392).
[W825 05:46:16.252011905 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 404750336 bytes (free: 7602176, total: 150110011392).


OOM during training-step | recovered=True

Conditioning length 256
  physical microbatch 1: PASS | allocated 65.68 GiB | reserved 73.70 GiB
  physical microbatch 2: PASS | allocated 68.08 GiB | reserved 79.05 GiB
  physical microbatch 4: PASS | allocated 78.23 GiB | reserved 86.74 GiB
  physical microbatch 8: PASS | allocated 98.69 GiB | reserved 101.63 GiB
  physical microbatch 16: 

[W825 05:46:25.830731841 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 1023410176 bytes (free: 714342400, total: 150110011392).
[W825 05:46:25.945316348 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 228589568 bytes (free: 169082880, total: 150110011392).
[W825 05:46:25.946349252 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 228589568 bytes (free: 169082880, total: 150110011392).


OOM during training-step | recovered=True

Conditioning length 512
  physical microbatch 1: PASS | allocated 65.68 GiB | reserved 74.94 GiB
  physical microbatch 2: PASS | allocated 70.32 GiB | reserved 81.41 GiB
  physical microbatch 4: PASS | allocated 82.77 GiB | reserved 91.60 GiB
  physical microbatch 8: PASS | allocated 107.83 GiB | reserved 111.49 GiB
  physical microbatch 16: 

...

OOM during training-step | recovered=True

Conditioning length 4,096
  physical microbatch 1: PASS | allocated 79.94 GiB | reserved 88.32 GiB
  physical microbatch 2: PASS | allocated 102.18 GiB | reserved 104.73 GiB
  physical microbatch 4: 

[W825 05:46:59.020625104 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 247463936 bytes (free: 108265472, total: 150110011392).
[W825 05:46:59.124164585 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 247463936 bytes (free: 208928768, total: 150110011392).


OOM during training-step | recovered=True

Conditioning length 8,192
  physical microbatch 1: PASS | allocated 98.15 GiB | reserved 101.33 GiB
  physical microbatch 2: 

[W825 05:47:03.084380446 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 895483904 bytes (free: 120848384, total: 150110011392).
[W825 05:47:03.246263400 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 895483904 bytes (free: 125042688, total: 150110011392).


OOM during training-step | recovered=True

Attempted 30 FLUX cases


## 9. Summarize the durable result

The largest PASS is a measured capacity point. The largest safe batch is more
conservative: it must also keep peak reserved memory at or below 90% of total
H200 VRAM. If every configured batch passes, the result is a lower bound rather
than the physical maximum.


In [12]:
SUMMARY_BY_LENGTH = []
for sequence_length in SEQUENCE_LENGTHS:
    length_cases = [
        record
        for record in CASE_RESULTS
        if record['sequence_length'] == sequence_length
    ]
    passing_cases = [
        record for record in length_cases if record['status'] == 'PASS'
    ]
    safe_cases = [
        record
        for record in passing_cases
        if record['within_safe_reserved_fraction']
    ]
    oom_cases = [
        record for record in length_cases if record['status'] == 'OOM'
    ]
    configured_max_tested = (
        max(
            record['physical_microbatch_size']
            for record in length_cases
        )
        if length_cases
        else None
    )
    largest_passing = (
        max(
            record['physical_microbatch_size']
            for record in passing_cases
        )
        if passing_cases
        else None
    )
    largest_safe = (
        max(
            record['physical_microbatch_size']
            for record in safe_cases
        )
        if safe_cases
        else None
    )
    SUMMARY_BY_LENGTH.append(
        {
            'sequence_length': sequence_length,
            'largest_passing_microbatch': largest_passing,
            'largest_safe_microbatch': largest_safe,
            'first_oom_microbatch': (
                oom_cases[0]['physical_microbatch_size']
                if oom_cases
                else None
            ),
            'largest_configured_batch_was_tested': (
                configured_max_tested == FLUX_MICROBATCH_SIZES[-1]
            ),
            'capacity_is_lower_bound': (
                largest_passing == FLUX_MICROBATCH_SIZES[-1]
            ),
        }
    )

REPORT['summary_by_length'] = SUMMARY_BY_LENGTH
REPORT['sweep_aborted'] = SWEEP_ABORTED
REPORT['status'] = 'INCOMPLETE' if SWEEP_ABORTED else 'COMPLETE'
REPORT['completed_at_utc'] = datetime.now(timezone.utc).isoformat()
save_report()

print(f"Status: {REPORT['status']}")
print(f'Report: {REPORT_PATH}')
for summary in SUMMARY_BY_LENGTH:
    print(summary)
SUMMARY_BY_LENGTH


Status: COMPLETE
Report: /mnt/diffusion-arc-models/results/single-gpu-memory-footprint/20260825T054229658645Z/report.json
{'sequence_length': 128, 'largest_passing_microbatch': 16, 'largest_safe_microbatch': 8, 'first_oom_microbatch': 32, 'largest_configured_batch_was_tested': False, 'capacity_is_lower_bound': False}
{'sequence_length': 256, 'largest_passing_microbatch': 8, 'largest_safe_microbatch': 8, 'first_oom_microbatch': 16, 'largest_configured_batch_was_tested': False, 'capacity_is_lower_bound': False}
{'sequence_length': 512, 'largest_passing_microbatch': 8, 'largest_safe_microbatch': 8, 'first_oom_microbatch': 16, 'largest_configured_batch_was_tested': False, 'capacity_is_lower_bound': False}
{'sequence_length': 1024, 'largest_passing_microbatch': 8, 'largest_safe_microbatch': 4, 'first_oom_microbatch': 16, 'largest_configured_batch_was_tested': False, 'capacity_is_lower_bound': False}
{'sequence_length': 2048, 'largest_passing_microbatch': 4, 'largest_safe_microbatch': 4, 'fi

[{'sequence_length': 128,
  'largest_passing_microbatch': 16,
  'largest_safe_microbatch': 8,
  'first_oom_microbatch': 32,
  'largest_configured_batch_was_tested': False,
  'capacity_is_lower_bound': False},
 {'sequence_length': 256,
  'largest_passing_microbatch': 8,
  'largest_safe_microbatch': 8,
  'first_oom_microbatch': 16,
  'largest_configured_batch_was_tested': False,
  'capacity_is_lower_bound': False},
 {'sequence_length': 512,
  'largest_passing_microbatch': 8,
  'largest_safe_microbatch': 8,
  'first_oom_microbatch': 16,
  'largest_configured_batch_was_tested': False,
  'capacity_is_lower_bound': False},
 {'sequence_length': 1024,
  'largest_passing_microbatch': 8,
  'largest_safe_microbatch': 4,
  'first_oom_microbatch': 16,
  'largest_configured_batch_was_tested': False,
  'capacity_is_lower_bound': False},
 {'sequence_length': 2048,
  'largest_passing_microbatch': 4,
  'largest_safe_microbatch': 4,
  'first_oom_microbatch': 8,
  'largest_configured_batch_was_tested': Fa

## Interpretation and next decision

This notebook measures one-step capacity, not long-run stability, learning
quality, or throughput. Dummy token content is shape-faithful but not a
representative ARC length distribution. CUDA allocator state after an OOM can
also make a same-kernel boundary conservative.

Use the largest safe microbatch as the initial batch-size candidate, then rerun
that case and its immediate larger neighbor in fresh kernels. A later,
separately approved experiment can add activation checkpointing as an explicit
on/off dimension once it is integrated into the training configuration.
